In [0]:
from pyspark.sql.functions import current_timestamp
import requests
from datetime import date, timedelta

spark.sql("CREATE SCHEMA IF NOT EXISTS rocket_lab.bronze")

caminho_base = "/Volumes/rocket_lab/default/rocket_lab/"
arquivos = {
    "movies_info_TMDB_IMDB.csv": "tb_movies_info",
    "movies_financials_IMDB_TMDB.csv": "tb_movies_financials",
    "movies_metrics_IMDB_TMDB.csv": "tb_movies_metrics",
    "credits_and_tags_IMDB_TMDB.csv": "tb_credits_and_tags",
    "movies_reviews.csv": "tb_movies_reviews"
}

for arquivo, tabela in arquivos.items():
    df = spark.read.csv(f"{caminho_base}{arquivo}", header=True)
    df = df.withColumn("ingestion_datetime", current_timestamp())   # como usamos dfs diferentes em cada iteracao, ignoro esse warning
    df.write.mode("append").saveAsTable(f"rocket_lab.bronze.{tabela}")

dbutils.widgets.text("data_inicio", "")
dbutils.widgets.text("data_fim", "")

data_inicio = dbutils.widgets.get("data_inicio")
data_fim = dbutils.widgets.get("data_fim")

if not data_inicio:
    data_inicio = (date.today() - timedelta(days=7)).strftime("%m-%d-%Y")
if not data_fim:
    data_fim = date.today().strftime("%m-%d-%Y")

url = (
    "https://olinda.bcb.gov.br/olinda/servico/PTAX/versao/v1/odata/"
    "CotacaoDolarPeriodo(dataInicial=@dataInicial,dataFinalCotacao=@dataFinalCotacao)"
    f"?@dataInicial='{data_inicio}'&@dataFinalCotacao='{data_fim}'"
    "&$select=dataHoraCotacao,cotacaoCompra&$format=json"
)

resposta = requests.get(url).json()
df_cotacao = spark.createDataFrame(resposta["value"])
df_cotacao.write.mode("append").saveAsTable("rocket_lab.bronze.tb_cotacao_dolar")

In [0]:
%sql
-- limpeza ou verificação das tabelas
-- DROP SCHEMA rocket_lab.bronze CASCADE; -- limpa tudo
SHOW TABLES IN rocket_lab.bronze;

In [0]:
%sql
-- verificação da cotação
SELECT * FROM rocket_lab.bronze.tb_cotacao_dolar;

In [0]:
%sql
-- verificação das tabelas via tamanho
SELECT 'tb_credits_and_tags' AS tabela, COUNT(*) AS linhas FROM rocket_lab.bronze.tb_credits_and_tags
UNION ALL SELECT 'tb_movies_financials', COUNT(*) FROM rocket_lab.bronze.tb_movies_financials
UNION ALL SELECT 'tb_movies_info', COUNT(*) FROM rocket_lab.bronze.tb_movies_info
UNION ALL SELECT 'tb_movies_metrics', COUNT(*) FROM rocket_lab.bronze.tb_movies_metrics
UNION ALL SELECT 'tb_movies_reviews', COUNT(*) FROM rocket_lab.bronze.tb_movies_reviews